This code needs to run with a custom environment as analysis3-unstable does not have NEMOSIS.

In [1]:
import numpy as np, pandas as pd, os
from pathlib import Path

from nemosis import dynamic_data_compiler

In [2]:
os.chdir('/g/data/ng72/ms5578/ID_HW_BARRA')
working_dir = Path().absolute()

ehf_fpath = '/scratch/ng72/ms5578'
nmap_path = '/g/data/ng72/ms5578/ID_HW_BARRA/data/raw'
write_path = '/scratch/ng72/ms5578/time_series/generation_18_19'
raw_NEM_cache = '/scratch/ng72/ms5578/nemosis_cache'

In [3]:
sdate, edate = "2018/10/01 00:00:00", "2019/03/31 23:59:59"

In [4]:
gen_df = dynamic_data_compiler(start_time=sdate,
                                   end_time=edate,
                                   table_name='DISPATCHLOAD',
                                   raw_data_location=raw_NEM_cache,
                                   fformat="feather")

gen_df = gen_df.rename(columns={'SETTLEMENTDATE': "time"})
gen_df = gen_df.reset_index(drop=True)

INFO: Compiling data for table DISPATCHLOAD
INFO: Creating feather file for DISPATCHLOAD, 2018, 09
INFO: Creating feather file for DISPATCHLOAD, 2018, 10
INFO: Creating feather file for DISPATCHLOAD, 2018, 11
INFO: Creating feather file for DISPATCHLOAD, 2018, 12
INFO: Creating feather file for DISPATCHLOAD, 2019, 01
INFO: Creating feather file for DISPATCHLOAD, 2019, 02
INFO: Creating feather file for DISPATCHLOAD, 2019, 03
INFO: Returning DISPATCHLOAD.


In [5]:
gen_df['time'] = pd.to_datetime(gen_df['time'] )
gen_df = gen_df.set_index('time')

In [6]:
gen_df['INITIALMW'] = gen_df['INITIALMW'].apply(lambda x: x*5/60)
gen_df = gen_df.rename(columns={"INITIALMW": "TOTALMWh"}).reset_index()

In [7]:
for duid, df_duid in gen_df.groupby("DUID"):
    safe_duid = str(duid).replace("/", "_").replace("\\", "_")
    df_duid.to_csv(f"{write_path}/{safe_duid}.csv", index=False)